# soft_prompt_check — consolidated results for the soft-prompting arm
Reads the frozen splits and `scores/` only; generates nothing. The main score-derived arm-C result tables and diagnostics are reproduced here from the frozen score files; no generation or judging is rerun.

Three variants, all evaluated at step 2000: (i) learnable rows prepended at the MLLM input embeddings, (ii) learnable rows appended to the MLLM output, (iii) learnable K/V at every MLLM layer via `past_key_values`. Variants (i) and (ii) share P=668, holding prompt length and trainable-parameter count constant, but variant (ii) carries the append-after-padding training defect, so the injection-site contrast is compromised and is not interpreted causally; (iii) at p=139 is the parameter-matched point against LoRA rank 4.

## 1. Load and integrity

In [ ]:
from google.colab import drive
import os, json, math, re
from collections import defaultdict
import numpy as np
from scipy import stats
drive.mount("/content/drive")

DRIVE  = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES = f"{DRIVE}/scores"

def load_by_uid(path):
    d = {}
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: continue
        d[r["uid"]] = r
    return d

ov  = lambda r: math.sqrt(max(r["PF"], 0) * max(r["SC"], 0))
MET = [("PF", lambda r: r["PF"]), ("SC", lambda r: r["SC"]), ("Overall", ov)]

def guessed(path):
    n = 0
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if str(r.get("PF_reason", "")).startswith("guess_if_cannot_parse") \
           or str(r.get("SC_reason", "")).startswith("guess_if_cannot_parse"): n += 1
    return n

V     = {"i": "sp_i_p668", "ii": "sp_ii_p668", "iii": "sp_iii_p139"}
STEP  = 2000
VSTEPS = (500, 1000, 2000, 3000)

BASE = load_by_uid(f"{SCORES}/test__baseline.jsonl")
SP   = {v: load_by_uid(f"{SCORES}/test__{r}_step{STEP}.jsonl") for v, r in V.items()}
L4   = load_by_uid(f"{SCORES}/test__lora_r4_step1000.jsonl")   # LoRA r4 comparator; parameter-matched only to variant (iii) at 100.3%
L8   = load_by_uid(f"{SCORES}/test__lora_r8_step1000.jsonl")

VBASE = load_by_uid(f"{SCORES}/val__baseline_val.jsonl")
VAL   = {v: {s: load_by_uid(f"{SCORES}/val__{r}_step{s}.jsonl") for s in VSTEPS} for v, r in V.items()}

META = {r["output_image"] + "|" + "|".join(sorted(r["input_images"])): r
        for r in [json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]}

I  = sorted(set(BASE) & set(L4) & set(L8) & set.intersection(*[set(d) for d in SP.values()]))
IV = sorted(set(VBASE) & set.intersection(*[set(d) for r in VAL.values() for d in r.values()]))

files = ["test__baseline", "test__lora_r4_step1000", "test__lora_r8_step1000"] \
        + [f"test__{r}_step{STEP}" for r in V.values()] \
        + [f"val__{r}_step{s}" for r in V.values() for s in VSTEPS] + ["val__baseline_val"]
print(f"test rows paired across baseline, all three variants and both adapters : {len(I)}")
print(f"val rows paired across all variants and checkpoints                    : {len(IV)}")
print(f"guessed judge scores across every file above                           : {sum(guessed(f'{SCORES}/{f}.jsonl') for f in files)}")

Mounted at /content/drive
test rows paired across baseline, all three variants and both adapters : 488
val rows paired across all variants and checkpoints                    : 98
guessed judge scores across every file above                           : 0


## 2. Headline — test set, 488 rows

In [ ]:
def paired(A, B, idx, f):
    d = np.array([f(A[k]) - f(B[k]) for k in idx], float)
    n = len(d); se = d.std(ddof=1) / math.sqrt(n)
    p = stats.ttest_rel([f(A[k]) for k in idx], [f(B[k]) for k in idx])[1]
    tc = stats.t.ppf(0.95, n - 1)
    return dict(n=n, mean=d.mean(), sd=d.std(ddof=1), se=se, p=p,
                lo=d.mean() - tc * se, hi=d.mean() + tc * se, d=d)

def show(lab, A, B, idx, mets=MET):
    for name, f in mets:
        s = paired(A, B, idx, f)
        print(f"  {lab:32s} {name:8s} n={s['n']:4d}  delta {s['mean']:+7.3f}  sigma_d {s['sd']:5.3f}  "
              f"SE {s['se']:.3f}  p {s['p']:.3g}  90%CI [{s['lo']:+.3f}, {s['hi']:+.3f}]")

BOUND = 0.15
def tost(A, B, idx, bound=BOUND):
    s = paired(A, B, idx, ov); n = s["n"]
    p_lo = 1 - stats.t.cdf((s["mean"] + bound) / s["se"], n - 1)
    p_hi = stats.t.cdf((s["mean"] - bound) / s["se"], n - 1)
    return s, p_lo, p_hi, (p_lo < 0.05 and p_hi < 0.05)

def delta_icc(A, B, idx):
    g = defaultdict(list)
    for k in idx: g[k.split("|")[0]].append(ov(A[k]) - ov(B[k]))
    allv = np.concatenate([np.array(v) for v in g.values()]); kbar = np.mean([len(v) for v in g.values()])
    grand = allv.mean()
    msb = sum(len(v) * (np.mean(v) - grand)**2 for v in g.values()) / (len(g) - 1)
    msw = sum(((np.array(v) - np.mean(v))**2).sum() for v in g.values()) / (len(allv) - len(g))
    return (msb - msw) / (msb + (kbar - 1) * msw)

def cluster_se(A, B, idx):
    d = np.array([ov(A[k]) - ov(B[k]) for k in idx]); m = d.mean(); n = len(d)
    g = defaultdict(list)
    for k, x in zip(idx, d): g[k.split("|")[0]].append(x)
    G = len(g)
    num = sum((np.sum(np.array(v) - m))**2 for v in g.values())
    return m, d.std(ddof=1) / math.sqrt(n), math.sqrt(num / n**2 * (G / (G - 1))), G

PERSON = {"man","woman","men","women","person","people","boy","girl","boys","girls","guy","lady","businessman",
          "businesswoman","musician","musicians","child","children","kid","kids","baby","model","player","athlete",
          "worker","couple","gentleman","barber","tourist","tourists","doctor","doctors","assistant","assistants"}
_PREP = (r"\b(in|with|on|at|by|of|from|and|to|dressed|wearing|holding|using|reviewing|sitting|standing|"
         r"typing|sketching|working|reading|writing|playing|looking|smiling|drinking|reaching|leaning|posing|walking|browsing)\b")
def person_obj(instr):
    kinds = set()
    for p in re.split(r"(?<=\.)\s+", instr):
        if "<img>" not in p: continue
        m = re.search(r"(?:the\s+)?(.+?)\s+(?:is|are)\s+", p.lower())
        head = re.split(_PREP, m.group(1) if m else p.lower())[0].strip().split()
        kinds.add("person" if (head and head[-1] in PERSON) else "object")
    return "person" if kinds == {"person"} else ("object" if kinds == {"object"} else "mixed")

def strata(A, B, key, order, idx, PROV):
    print(f"  {key:12s} {'n':>4s} {'dPF':>8s} {'dSC':>8s} {'dOverall':>9s} {'p':>8s}")
    for lev in order:
        u = [k for k in idx if (PROV[k] if key == "provenance" else B[k][key]) == lev]
        if not u: continue
        dP = np.mean([A[k]["PF"] - B[k]["PF"] for k in u])
        dS = np.mean([A[k]["SC"] - B[k]["SC"] for k in u])
        d  = np.array([ov(A[k]) - ov(B[k]) for k in u])
        p  = stats.ttest_rel([ov(A[k]) for k in u], [ov(B[k]) for k in u])[1]
        print(f"  {str(lev):12s} {len(u):>4d} {dP:>+8.3f} {dS:>+8.3f} {d.mean():>+9.3f} {p:>8.3f}")

def agg(d, idx): return tuple(np.mean([f(d[k]) for k in idx]) for _, f in MET)
print(f"{'run':34s} {'PF':>7s} {'SC':>7s} {'Overall':>7s}")
print(f"{'prompting baseline':34s} " + " ".join(f"{x:7.3f}" for x in agg(BASE, I)))
print(f"{'LoRA r4 @1000 (comparator)':34s} " + " ".join(f"{x:7.3f}" for x in agg(L4, I)))
print(f"{'LoRA r8 @1000':34s} " + " ".join(f"{x:7.3f}" for x in agg(L8, I)))
for v in ("iii", "ii", "i"):
    print(f"{f'soft prompt ({v}) @{STEP}':34s} " + " ".join(f"{x:7.3f}" for x in agg(SP[v], I)))

run                                     PF      SC Overall
prompting baseline                   8.004   8.088   7.963
LoRA r4 @1000 (comparator)           8.027   7.855   7.848
LoRA r8 @1000                        8.064   7.846   7.843
soft prompt (iii) @2000              7.830   6.859   7.137
soft prompt (ii) @2000               7.832   6.834   7.114
soft prompt (i) @2000                7.123   6.578   6.612


## 3. Parameter accounting
Trainable counts are reported explicitly because only (iii) is parameter-matched to LoRA r4 at 100.3%; (i) and (ii) are 53.6%-capacity comparisons, so the hierarchy is not read at matched capacity throughout. (i) and (ii) hold P=668 rows of width 2048; (iii) holds `layers x 2 x kv_heads x p x head_dim`.

In [ ]:
HID, LAYERS, KV, HEAD_DIM = 2048, 36, 2, 128
P_SHALLOW, P_PREFIX = 668, 139
n_i   = P_SHALLOW * HID
n_iii = LAYERS * 2 * KV * P_PREFIX * HEAD_DIM
n_r4  = 38 * 16800 * 4
for lab, n in [("(i)  MLLM input, P=668", n_i), ("(ii) MLLM output, P=668", n_i),
               ("(iii) prefix K/V, p=139", n_iii), ("LoRA r4", n_r4)]:
    print(f"  {lab:26s} {n:>10,}  = {100*n/n_r4:6.1f}% of LoRA r4")

  (i)  MLLM input, P=668      1,368,064  =   53.6% of LoRA r4
  (ii) MLLM output, P=668     1,368,064  =   53.6% of LoRA r4
  (iii) prefix K/V, p=139     2,562,048  =  100.3% of LoRA r4
  LoRA r4                     2,553,600  =  100.0% of LoRA r4


## 4. Paired contrasts — against the baseline, against the LoRA r4 comparator, between variants
Selection note: for variant (iii) the exclusion of the warmup-boundary checkpoint was a documented post-hoc eligibility change made after inspecting the validation curve — unrestricted argmax is step 500, post-warmup argmax is step 2000. For variants (i) and (ii) the two rules coincide.

In [ ]:
print("against the prompting baseline:")
for v in ("i", "ii", "iii"): show(f"({v}) @{STEP} - baseline", SP[v], BASE, I)
print("\nagainst LoRA r4 @1000 — parameter-matched only for (iii) at 100.3%; (i) and (ii) are lower-capacity comparisons at 53.6% (the hierarchy claim):")
for v in ("i", "ii", "iii"): show(f"({v}) - LoRA r4", SP[v], L4, I)
print("\nbetween variants:")
show("(i) - (ii)  [same P, diff site]", SP["i"], SP["ii"], I)
show("(iii) - (ii) [depth vs width]",      SP["iii"], SP["ii"], I)
show("(iii) - (i)",                        SP["iii"], SP["i"],  I)

against the prompting baseline:
  (i) @2000 - baseline             PF       n= 488  delta  -0.881  sigma_d 1.942  SE 0.088  p 1.28e-21  90%CI [-1.026, -0.736]
  (i) @2000 - baseline             SC       n= 488  delta  -1.510  sigma_d 2.662  SE 0.121  p 2.03e-31  90%CI [-1.709, -1.312]
  (i) @2000 - baseline             Overall  n= 488  delta  -1.351  sigma_d 2.225  SE 0.101  p 4.15e-35  90%CI [-1.517, -1.185]
  (ii) @2000 - baseline            PF       n= 488  delta  -0.172  sigma_d 1.493  SE 0.068  p 0.0112  90%CI [-0.284, -0.061]
  (ii) @2000 - baseline            SC       n= 488  delta  -1.254  sigma_d 2.578  SE 0.117  p 2.56e-24  90%CI [-1.446, -1.062]
  (ii) @2000 - baseline            Overall  n= 488  delta  -0.849  sigma_d 1.974  SE 0.089  p 9.36e-20  90%CI [-0.996, -0.702]
  (iii) @2000 - baseline           PF       n= 488  delta  -0.174  sigma_d 1.546  SE 0.070  p 0.0132  90%CI [-0.290, -0.059]
  (iii) @2000 - baseline           SC       n= 488  delta  -1.230  sigma_d 2.330  S

## 5. TOST against the pre-registered ±0.15 bound
Equivalence is declared only if both one-sided tests reject, i.e. the 90% CI lies entirely inside the bound. A CI straddling the bound is inconclusive, not equivalent.

In [ ]:
for lab, A, B in ([(f"({v}) vs baseline", SP[v], BASE) for v in ("i","ii","iii")]
                  + [(f"({v}) vs LoRA r4", SP[v], L4) for v in ("i","ii","iii")]):
    s, p_lo, p_hi, eq = tost(A, B, I)
    print(f"  {lab:22s} delta {s['mean']:+.3f}  90%CI [{s['lo']:+.3f}, {s['hi']:+.3f}]  "
          f"p(>-{BOUND})={p_lo:.3f}  p(<+{BOUND})={p_hi:.2g}  -> {'equivalent' if eq else 'not equivalent'}")

  (i) vs baseline        delta -1.351  90%CI [-1.517, -1.185]  p(>-0.15)=1.000  p(<+0.15)=6.4e-42  -> not equivalent
  (ii) vs baseline       delta -0.849  90%CI [-0.996, -0.702]  p(>-0.15)=1.000  p(<+0.15)=2.7e-26  -> not equivalent
  (iii) vs baseline      delta -0.826  90%CI [-0.963, -0.688]  p(>-0.15)=1.000  p(<+0.15)=2.7e-28  -> not equivalent
  (i) vs LoRA r4         delta -1.236  90%CI [-1.390, -1.082]  p(>-0.15)=1.000  p(<+0.15)=1.7e-41  -> not equivalent
  (ii) vs LoRA r4        delta -0.734  90%CI [-0.875, -0.594]  p(>-0.15)=1.000  p(<+0.15)=3.2e-23  -> not equivalent
  (iii) vs LoRA r4       delta -0.711  90%CI [-0.837, -0.585]  p(>-0.15)=1.000  p(<+0.15)=9.4e-27  -> not equivalent


## 6. Cluster-robust standard errors
Rows sharing a target scene are not independent. The delta ICC and a target-clustered SE bound how much the iid SE understates uncertainty.

In [ ]:
print(f"  {'contrast':24s} {'ICC(delta)':>10s} {'SE iid':>8s} {'SE clust':>9s} {'t iid':>7s} {'t clust':>8s} {'p clust':>8s}")
for lab, A, B in ([(f"({v}) - baseline", SP[v], BASE) for v in ("i","ii","iii")]
                  + [(f"({v}) - LoRA r4", SP[v], L4) for v in ("i","ii","iii")]):
    icc = delta_icc(A, B, I); m, se_i, se_c, G = cluster_se(A, B, I)
    p = 2 * (1 - stats.norm.cdf(abs(m / se_c)))
    print(f"  {lab:24s} {icc:>10.3f} {se_i:>8.4f} {se_c:>9.4f} {m/se_i:>7.2f} {m/se_c:>8.2f} {p:>8.3f}")

  contrast                 ICC(delta)   SE iid  SE clust   t iid  t clust  p clust
  (i) - baseline                0.134   0.1007    0.1368  -13.41    -9.87    0.000
  (ii) - baseline               0.063   0.0894    0.1037   -9.50    -8.19    0.000
  (iii) - baseline              0.046   0.0835    0.0945   -9.89    -8.74    0.000
  (i) - LoRA r4                 0.113   0.0936    0.1233  -13.20   -10.02    0.000
  (ii) - LoRA r4                0.050   0.0852    0.0947   -8.62    -7.75    0.000
  (iii) - LoRA r4               0.049   0.0762    0.0865   -9.33    -8.22    0.000


## 7. Stratification
Only on variables fixed before generation. Provenance is derived from the instruction, never from a score.

In [ ]:
PROV = {k: person_obj(META[k]["instruction"]) for k in I}
for v in ("i", "ii", "iii"):
    print(f"variant ({v}) vs baseline:")
    strata(SP[v], BASE, "ref_kind",   ["no_up","mixed","all_up"], I, PROV)
    strata(SP[v], BASE, "n_refs",     [1,2,3],                    I, PROV)
    strata(SP[v], BASE, "provenance", ["person","object","mixed"],I, PROV)
    print()

variant (i) vs baseline:
  ref_kind        n      dPF      dSC  dOverall        p
  no_up         157   -1.395   -1.312    -1.398    0.000
  mixed         133   -0.571   -0.293    -0.436    0.001
  all_up        198   -0.682   -2.485    -1.928    0.000
  n_refs          n      dPF      dSC  dOverall        p
  1             243   -1.029   -1.992    -1.772    0.000
  2             194   -0.732   -1.124    -0.985    0.000
  3              51   -0.745   -0.686    -0.735    0.009
  provenance      n      dPF      dSC  dOverall        p
  person        130   -1.246   -1.008    -1.171    0.000
  object        224   -0.862   -2.540    -2.012    0.000
  mixed         134   -0.560   -0.276    -0.421    0.001

variant (ii) vs baseline:
  ref_kind        n      dPF      dSC  dOverall        p
  no_up         157   -0.236   -0.892    -0.593    0.000
  mixed         133   -0.248   -0.316    -0.287    0.010
  all_up        198   -0.071   -2.172    -1.430    0.000
  n_refs          n      dPF      dS

## 8. Validation curves and the two stopping signals
Diffusion val loss is transcribed from each run's `val_loss.jsonl`. Selection is the GPT-4.1 argmax over the post-warmup checkpoints; `warmup_t` is 500, so step 500 is the warmup-boundary checkpoint. For variant (iii) it is also the unrestricted argmax, and a documented post-hoc eligibility restriction excludes it, selecting the post-warmup argmax at step 2000. Prefix-norm drift is already +119% at step 500, so its score is not explained by the checkpoint being unadapted.

In [ ]:
VLOSS = {  # soft_prompt/experiments/<run>/val_loss.jsonl, every 250 steps
 "i":   {250:0.35715, 500:0.34758, 750:0.34642, 1000:0.34585, 1250:0.34559, 1500:0.34616,
         1750:0.34565, 2000:0.34507, 2250:0.34535, 2500:0.34531, 2750:0.34460, 3000:0.34490},
 "ii":  {250:0.34340, 500:0.34295, 750:0.34279, 1000:0.34233, 1250:0.34232, 1500:0.34219,
         1750:0.34212, 2000:0.34197, 2250:0.34256, 2500:0.34193, 2750:0.34184, 3000:0.34180},
 "iii": {250:0.34858, 500:0.34605, 750:0.34432, 1000:0.34388, 1250:0.34369, 1500:0.34352,
         1750:0.34341, 2000:0.34338, 2250:0.34345, 2500:0.34335, 2750:0.34320, 3000:0.34312}}

for v in ("i", "ii", "iii"):
    b = np.mean([ov(VBASE[k]) for k in IV])
    print(f"variant ({v}) — val baseline Overall {b:.3f}")
    print(f"  {'step':>5s} {'val loss':>9s} {'GPT Overall':>12s} {'GPT PF':>7s} {'GPT SC':>7s} {'vs base':>9s} {'p':>8s}")
    for s in VSTEPS:
        D = VAL[v][s]; d = np.array([ov(D[k]) - ov(VBASE[k]) for k in IV])
        print(f"  {s:>5d} {VLOSS[v][s]:>9.5f} {np.mean([ov(D[k]) for k in IV]):>12.3f} "
              f"{np.mean([D[k]['PF'] for k in IV]):>7.3f} {np.mean([D[k]['SC'] for k in IV]):>7.3f} "
              f"{d.mean():>+9.3f} {stats.ttest_1samp(d, 0)[1]:>8.3g}")
    g = {s: np.mean([ov(VAL[v][s][k]) for k in IV]) for s in VSTEPS}
    post = [s for s in VSTEPS if s > 500]
    grid = {s: VLOSS[v][s] for s in VSTEPS}
    print(f"  GPT argmax (all) {max(g, key=g.get)} | GPT argmax (post-warmup) {max(post, key=lambda s: g[s])} | "
          f"val-loss argmin: scored grid {min(grid, key=grid.get)}, full curve {min(VLOSS[v], key=VLOSS[v].get)}")
    rho = stats.spearmanr([g[s] for s in VSTEPS], [-VLOSS[v][s] for s in VSTEPS]).statistic
    print(f"  Spearman(GPT Overall, -val loss) = {rho:+.2f}\n")

variant (i) — val baseline Overall 7.739
   step  val loss  GPT Overall  GPT PF  GPT SC   vs base        p
    500   0.34758        6.311   6.551   6.347    -1.427  2.7e-09
   1000   0.34585        6.566   6.898   6.449    -1.173 1.13e-06
   2000   0.34507        6.661   7.020   6.633    -1.078 2.17e-06
   3000   0.34490        6.561   6.878   6.480    -1.178 1.72e-07
  GPT argmax (all) 2000 | GPT argmax (post-warmup) 2000 | val-loss argmin: scored grid 3000, full curve 2750
  Spearman(GPT Overall, -val loss) = +0.40

variant (ii) — val baseline Overall 7.739
   step  val loss  GPT Overall  GPT PF  GPT SC   vs base        p
    500   0.34295        6.843   7.408   6.520    -0.895 8.85e-05
   1000   0.34233        6.897   7.500   6.612    -0.842 0.000206
   2000   0.34197        6.960   7.531   6.765    -0.779 0.000421
   3000   0.34180        6.877   7.469   6.704    -0.862 9.71e-05
  GPT argmax (all) 2000 | GPT argmax (post-warmup) 2000 | val-loss argmin: scored grid 3000, full curve 

## 9. Val-to-test transfer

In [ ]:
print(f"  {'contrast':26s} {'val (n=98)':>12s} {'test (n=488)':>14s} {'same sign':>10s}")
for v in ("i", "ii", "iii"):
    dv = np.mean([ov(VAL[v][STEP][k]) - ov(VBASE[k]) for k in IV])
    dt = np.mean([ov(SP[v][k]) - ov(BASE[k]) for k in I])
    print(f"  {f'({v}) @{STEP} - baseline':26s} {dv:>+12.3f} {dt:>+14.3f} {str(np.sign(dv)==np.sign(dt)):>10s}")
ntg = len({k.split("|")[0] for k in IV})
print(f"\n  val slice: {len(IV)} rows over {ntg} targets ({len(IV)/ntg:.1f} rows/target)")
print("  step 2000 was selected on val, so its val estimate carries selection bias; the test split sizes the effect")

  contrast                     val (n=98)   test (n=488)  same sign
  (i) @2000 - baseline             -1.078         -1.351       True
  (ii) @2000 - baseline            -0.779         -0.849       True
  (iii) @2000 - baseline           -0.666         -0.826       True

  val slice: 98 rows over 20 targets (4.9 rows/target)
  step 2000 was selected on val, so its val estimate carries selection bias; the test split sizes the effect


## 10. Where the damage sits
The arm's mechanistic result. References never enter the MLLM text path, so a learned prompt cannot add evidence about a subject; it can only perturb the conditioning the diffusion decoder receives.

In [ ]:
print(f"  {'variant':10s} {'dPF':>8s} {'dSC':>8s} {'dOverall':>9s} {'SC share of the loss':>22s}")
for v in ("i", "ii", "iii"):
    dP = np.mean([SP[v][k]["PF"] - BASE[k]["PF"] for k in I])
    dS = np.mean([SP[v][k]["SC"] - BASE[k]["SC"] for k in I])
    dO = np.mean([ov(SP[v][k]) - ov(BASE[k]) for k in I])
    print(f"  {f'({v})':10s} {dP:>+8.3f} {dS:>+8.3f} {dO:>+9.3f} {abs(dS)/(abs(dP)+abs(dS)):>22.0%}")

  variant         dPF      dSC  dOverall   SC share of the loss
  (i)          -0.881   -1.510    -1.351                    63%
  (ii)         -0.172   -1.254    -0.849                    88%
  (iii)        -0.174   -1.230    -0.826                    88%


## 11. Verdict against the Petrov prediction
The falsifiable prediction: if consumer-electronics adaptation is elicitation of a latent capability, soft prompting approaches LoRA; if it requires learning, soft prompting stays systematically below.

In [ ]:
print(f"  {'variant':8s} {'vs LoRA r4':>12s} {'90% CI':>20s} {'p':>10s}  verdict")
for v in ("i", "ii", "iii"):
    s = paired(SP[v], L4, I, ov)
    below = s["hi"] < -BOUND
    ci = f"[{s['lo']:+.3f}, {s['hi']:+.3f}]"
    print(f"  {f'({v})':8s} {s['mean']:>+12.3f} {ci:>20s} {s['p']:>10.3g}  "
          f"{'systematically below the LoRA r4 comparator' if below else 'not resolved at this bound'}")
# SP-(iii) vs SP-(ii): the capacity contrast — (iii) is matched to LoRA r4 at 100.3% and holds
# 1.87x the trainable parameters of (ii). Reported iid and target-clustered,
# because the 488 rows come from 100 target scenes and this delta is clustered.
_m, _se, _secl, _G = cluster_se(SP["iii"], SP["ii"], I)
_icc = delta_icc(SP["iii"], SP["ii"], I)
_s, _plo, _phi, _eq = tost(SP["iii"], SP["ii"], I)
_zlo = 1 - stats.norm.cdf((_m + BOUND) / _secl); _zhi = stats.norm.cdf((_m - BOUND) / _secl)
_tlo = 1 - stats.t.cdf((_m + BOUND) / _secl, _G - 1); _thi = stats.t.cdf((_m - BOUND) / _secl, _G - 1)
print(f"\n  (iii) - (ii) = {_m:+.4f}  delta ICC {_icc:.3f} over {_G} target clusters")
print(f"    iid       SE {_se:.4f}  90% CI [{_m-1.645*_se:+.4f}, {_m+1.645*_se:+.4f}]  TOST p {max(_plo,_phi):.4f}  equivalent {_eq}")
print(f"    clustered SE {_secl:.4f}  90% CI [{_m-1.645*_secl:+.4f}, {_m+1.645*_secl:+.4f}]  "
      f"TOST p {max(_zlo,_zhi):.4f} (normal) / {max(_tlo,_thi):.4f} (t)  equivalent {max(_zlo,_zhi) < 0.05}")
print("    formal equivalence within the bound is not robust to target clustering; the capacity")
print("    subclaim should be stated as 'no difference detected', not as demonstrated equivalence")

print("\n  every variant sits below LoRA r4 with the 90% CI clear of the bound; the parameter-matched")
print("  case is (iii) - LoRA r4 = -0.711 Overall, 90% CI [-0.837, -0.585]")
print("  -> the prediction that soft prompting approaches LoRA is rejected: a parameter-matched")
print("     prefix method is substantially worse than the adapter, a method-class difference")
print("  -> this does not resolve Petrov's elicitation-vs-learning dichotomy, because LoRA r4 is")
print("     itself below the prompting baseline; neither arm achieved a useful adaptation here")

  variant    vs LoRA r4               90% CI          p  verdict
  (i)            -1.236     [-1.390, -1.082]   3.18e-34  systematically below the LoRA r4 comparator
  (ii)           -0.734     [-0.875, -0.594]   9.78e-17  systematically below the LoRA r4 comparator
  (iii)          -0.711     [-0.837, -0.585]   3.71e-19  systematically below the LoRA r4 comparator

  (iii) - (ii) = +0.0234  delta ICC 0.097 over 100 target clusters
    iid       SE 0.0712  90% CI [-0.0937, +0.1405]  TOST p 0.0380  equivalent True
    clustered SE 0.0812  90% CI [-0.1101, +0.1569]  TOST p 0.0594 (normal) / 0.0610 (t)  equivalent False
    formal equivalence within the bound is not robust to target clustering; the capacity
    subclaim should be stated as 'no difference detected', not as demonstrated equivalence

  every variant sits below LoRA r4 with the 90% CI clear of the bound; the parameter-matched
  case is (iii) - LoRA r4 = -0.711 Overall, 90% CI [-0.837, -0.585]
  -> the prediction that soft pro